# SNU AI Challenge - structured scorer without Transformer

This controlled five-fold experiment changes only the EXP-003 full-sequence verifier: `verifier="none"`. The original 9,535 rows, frozen feature cache, folds, resident candidate plans, batch sizes, optimizer, seed, and early-stopping policy remain unchanged. No augmented data is used.

## 1. Parameters

In [6]:
import os
from pathlib import Path

DRIVE_ROOT = Path(os.environ.get("SNU_DRIVE_ROOT", "/content/drive/MyDrive/snu ai"))
BUNDLE_PATH = DRIVE_ROOT / "snu_colab_full_train_bundle.tar"
CODE_BUNDLE_PATH = DRIVE_ROOT / "snu_colab_no_transformer_training_code.tar"
WORKDIR = Path(os.environ.get("SNU_WORKDIR", "/content/snu_ai"))
DRIVE_CACHE_SHARDS = DRIVE_ROOT / "cache/full_train_9535_shards"
DRIVE_RESULTS_ROOT = DRIVE_ROOT / "results/full_24class_no_transformer_resident_v1"
LOCAL_CACHE_ROOT = WORKDIR / "feature_cache/full_train_9535"

FOLDS_TO_RUN = [0, 1, 2, 3, 4]
TRAIN_FINAL_SCORER = True
GPU_CONFIG_CONFIRMED = True  # Review the detected GPU cell, then change to True.

HIDDEN_DIM = 256
VERIFIER = "none"
BATCH_SIZE = 128
EVALUATION_BATCH_SIZE = 256
MAX_EPOCHS = 20
PATIENCE = 4
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
TOP_H = 5
TOP_M = 8
ALPHA, BETA, GAMMA, DELTA = 0.1, 1.0, 1.0, 1.0
SEED = 42

assert FOLDS_TO_RUN and all(fold in range(5) for fold in FOLDS_TO_RUN)
assert len(FOLDS_TO_RUN) == len(set(FOLDS_TO_RUN))
assert VERIFIER == "none"
assert BATCH_SIZE == 128 and EVALUATION_BATCH_SIZE == 256

## 2. Mount Drive and refresh the no-Transformer code overlay

In [7]:
import importlib
import subprocess
import sys

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    drive.mount("/content/drive")
else:
    print("google.colab is unavailable; configured paths will be used directly.")

if not BUNDLE_PATH.is_file():
    raise FileNotFoundError(f"Training bundle not found: {BUNDLE_PATH}")
if not CODE_BUNDLE_PATH.is_file():
    raise FileNotFoundError(f"No-Transformer code bundle not found: {CODE_BUNDLE_PATH}")

WORKDIR.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
FULL_FOLDS_CSV = WORKDIR / "snuaichallenge_data/splits/full_folds_5.csv"
if not FULL_FOLDS_CSV.is_file():
    subprocess.run(
        ["tar", "-xf", str(BUNDLE_PATH), "-C", str(WORKDIR),
         "snuaichallenge_data/splits/full_folds_5.csv"],
        check=True,
    )
subprocess.run(
    ["tar", "-xf", str(CODE_BUNDLE_PATH), "-C", str(WORKDIR)],
    check=True,
)
for module_name in list(sys.modules):
    if module_name == "snu_ordering" or module_name.startswith("snu_ordering."):
        del sys.modules[module_name]
importlib.invalidate_caches()
sys.path.insert(0, str(WORKDIR))

print(f"Project root: {WORKDIR}")
print(f"Feature shards: {DRIVE_CACHE_SHARDS}")
print(f"Experiment results: {DRIVE_RESULTS_ROOT}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root: /content/snu_ai
Feature shards: /content/drive/MyDrive/snu ai/cache/full_train_9535_shards
Experiment results: /content/drive/MyDrive/snu ai/results/full_24class_no_transformer_resident_v1


## 3. Validate GPU configuration and original data

The experiment deliberately keeps EXP-003's batch sizes for a controlled comparison. Review the detected GPU and VRAM below. Training is blocked until `GPU_CONFIG_CONFIRMED=True` is set in the parameter cell.

In [8]:
import json
import random
from collections import Counter

import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before continuing.")

from snu_ordering.data import load_samples

samples = load_samples(
    FULL_FOLDS_CSV,
    WORKDIR / "snuaichallenge_data/train",
    require_labels=True,
    validate_images=False,
)
answer_counts = Counter(sample.rank_label for sample in samples)
fold_counts = Counter(sample.fold for sample in samples)
identity_count = answer_counts[(1, 2, 3, 4)]
assert len(samples) == 9535
assert len(answer_counts) == 24
assert identity_count == 1478
assert all(1905 <= fold_counts[fold] <= 1909 for fold in range(5))
assert all(sample.fold is not None for sample in samples)

gpu_properties = torch.cuda.get_device_properties(0)
gpu_review = {
    "torch": torch.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "vram_gib": gpu_properties.total_memory / 1024**3,
    "cuda_capability": torch.cuda.get_device_capability(0),
    "precision": "BF16 AMP" if torch.cuda.get_device_capability(0)[0] >= 8 else "FP16 AMP",
    "train_batch_size": BATCH_SIZE,
    "evaluation_batch_size": EVALUATION_BATCH_SIZE,
    "resident_original_samples": len(samples),
    "verifier": VERIFIER,
}
print(json.dumps(gpu_review, indent=2))
if not GPU_CONFIG_CONFIRMED:
    raise RuntimeError(
        "Review GPU/VRAM/batch settings above, then set GPU_CONFIG_CONFIRMED=True."
    )

{
  "torch": "2.11.0+cu128",
  "gpu": "NVIDIA RTX PRO 6000 Blackwell Server Edition",
  "vram_gib": 94.97076416015625,
  "cuda_capability": [
    12,
    0
  ],
  "precision": "BF16 AMP",
  "train_batch_size": 128,
  "evaluation_batch_size": 256,
  "resident_original_samples": 9535,
  "verifier": "none"
}


## 4. Restore the completed original feature cache

In [9]:
import tarfile
import time

from snu_ordering.qwen_representations import cache_path_for

def format_duration(seconds):
    seconds = max(0, int(round(seconds)))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"

LOCAL_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
cache_shards = sorted(DRIVE_CACHE_SHARDS.glob("cache_*.tar"))
if not cache_shards:
    raise FileNotFoundError(f"No original feature-cache shards found: {DRIVE_CACHE_SHARDS}")
restore_started = time.monotonic()
for shard_index, shard_path in enumerate(cache_shards, 1):
    with tarfile.open(shard_path, "r") as archive:
        archive.extractall(LOCAL_CACHE_ROOT)
    elapsed = time.monotonic() - restore_started
    rate = shard_index / elapsed if elapsed else 0.0
    remaining = len(cache_shards) - shard_index
    eta = remaining / rate if rate else 0.0
    print(
        f"cache shards {shard_index}/{len(cache_shards)} | remaining={remaining} | "
        f"elapsed={format_duration(elapsed)} | ETA={format_duration(eta)}",
        flush=True,
    )
missing_cache = [
    sample.sample_id for sample in samples
    if not cache_path_for(LOCAL_CACHE_ROOT, sample.sample_id).is_file()
]
if missing_cache:
    raise FileNotFoundError(
        f"Missing {len(missing_cache)} original cached representations; first IDs: {missing_cache[:5]}"
    )
print(f"original feature cache ready: {len(samples)}/{len(samples)}")

cache shards 1/48 | remaining=47 | elapsed=00:00:00 | ETA=00:00:01
cache shards 2/48 | remaining=46 | elapsed=00:00:00 | ETA=00:00:01
cache shards 3/48 | remaining=45 | elapsed=00:00:00 | ETA=00:00:01
cache shards 4/48 | remaining=44 | elapsed=00:00:00 | ETA=00:00:01
cache shards 5/48 | remaining=43 | elapsed=00:00:00 | ETA=00:00:01
cache shards 6/48 | remaining=42 | elapsed=00:00:00 | ETA=00:00:01
cache shards 7/48 | remaining=41 | elapsed=00:00:00 | ETA=00:00:01


/tmp/ipykernel_14809/782362471.py:19: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  archive.extractall(LOCAL_CACHE_ROOT)


cache shards 8/48 | remaining=40 | elapsed=00:00:00 | ETA=00:00:02
cache shards 9/48 | remaining=39 | elapsed=00:00:00 | ETA=00:00:01
cache shards 10/48 | remaining=38 | elapsed=00:00:00 | ETA=00:00:01
cache shards 11/48 | remaining=37 | elapsed=00:00:00 | ETA=00:00:01
cache shards 12/48 | remaining=36 | elapsed=00:00:00 | ETA=00:00:01
cache shards 13/48 | remaining=35 | elapsed=00:00:00 | ETA=00:00:01
cache shards 14/48 | remaining=34 | elapsed=00:00:00 | ETA=00:00:01
cache shards 15/48 | remaining=33 | elapsed=00:00:00 | ETA=00:00:01
cache shards 16/48 | remaining=32 | elapsed=00:00:00 | ETA=00:00:01
cache shards 17/48 | remaining=31 | elapsed=00:00:00 | ETA=00:00:01
cache shards 18/48 | remaining=30 | elapsed=00:00:01 | ETA=00:00:01
cache shards 19/48 | remaining=29 | elapsed=00:00:01 | ETA=00:00:01
cache shards 20/48 | remaining=28 | elapsed=00:00:01 | ETA=00:00:01
cache shards 21/48 | remaining=27 | elapsed=00:00:01 | ETA=00:00:01
cache shards 22/48 | remaining=26 | elapsed=00:00:

## 5. Prepare the same GPU-resident features and grouping plans

In [10]:
import gc

from snu_ordering.structured_scorer import LightweightStructuredScorer
from snu_ordering.structured_training import (
    StructuredTrainingConfig,
    prepare_structured_samples,
    train_prepared_structured_scorer,
)

DEVICE = torch.device("cuda")
torch.set_float32_matmul_precision("high")
training_config = StructuredTrainingConfig(
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    batch_size=BATCH_SIZE,
    evaluation_batch_size=EVALUATION_BATCH_SIZE,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    seed=SEED,
    top_h=TOP_H,
    top_m=TOP_M,
    alpha=ALPHA,
    beta=BETA,
    gamma=GAMMA,
    delta=DELTA,
    use_amp=True,
)
prepared_samples = prepare_structured_samples(
    samples, LOCAL_CACHE_ROOT, DEVICE, top_h=TOP_H
)
assert len(prepared_samples) == 9535

def seed_everything(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def make_no_transformer_scorer():
    scorer = LightweightStructuredScorer(
        hidden_dim=HIDDEN_DIM, verifier=VERIFIER
    ).to(DEVICE)
    assert scorer.verifier_type == "none"
    assert scorer.sequence_transformer is None
    assert scorer.sequence_head is None
    assert scorer.cls_token is None
    assert scorer.position_embedding is None
    return scorer

probe_scorer = make_no_transformer_scorer()
print("trainable parameters without Transformer:", sum(
    parameter.numel() for parameter in probe_scorer.parameters() if parameter.requires_grad
))
del probe_scorer
gc.collect()
torch.cuda.empty_cache()

prepared candidates: 500/9535
prepared candidates: 1000/9535
prepared candidates: 1500/9535
prepared candidates: 2000/9535
prepared candidates: 2500/9535
prepared candidates: 3000/9535
prepared candidates: 3500/9535
prepared candidates: 4000/9535
prepared candidates: 4500/9535
prepared candidates: 5000/9535
prepared candidates: 5500/9535
prepared candidates: 6000/9535
prepared candidates: 6500/9535
prepared candidates: 7000/9535
prepared candidates: 7500/9535
prepared candidates: 8000/9535
prepared candidates: 8500/9535
prepared candidates: 9000/9535
prepared candidates: 9500/9535
prepared candidates: 9535/9535
resident training data: features=1.13 GiB, candidate_plans=0.35 GiB on cuda
trainable parameters without Transformer: 2166274


## 6. Train the selected folds

In [11]:
def run_fold(fold):
    output_dir = DRIVE_RESULTS_ROOT / f"fold_{fold}"
    if (output_dir / "training_summary.json").is_file():
        print(f"fold {fold}: already completed, skipping")
        return
    train_samples = [item for item in prepared_samples if item.sample.fold != fold]
    validation_samples = [item for item in prepared_samples if item.sample.fold == fold]
    assert {item.sample.rank_label for item in train_samples} == set(answer_counts)
    assert {item.sample.rank_label for item in validation_samples} == set(answer_counts)
    seed_everything(SEED)
    scorer = make_no_transformer_scorer()
    print(
        f"fold {fold}: train={len(train_samples)}, validation={len(validation_samples)}, "
        f"batch={BATCH_SIZE}, eval_batch={EVALUATION_BATCH_SIZE}, verifier={VERIFIER}"
    )
    train_prepared_structured_scorer(
        scorer, train_samples, validation_samples, output_dir, training_config
    )
    del scorer
    gc.collect()
    torch.cuda.empty_cache()

for fold in FOLDS_TO_RUN:
    run_fold(fold)

fold 0: train=7628, validation=1907, batch=128, eval_batch=256, verifier=none


/content/snu_ai/snu_ordering/structured_training.py:376: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_fp16_scaler)


{"epoch": 1, "train_loss": 2.881842227649739, "samples": 1907, "exact_match_accuracy": 0.17566858940744626, "lightweight_top8_recall": 0.6984792868379653}
{"epoch": 2, "train_loss": 2.673712183807805, "samples": 1907, "exact_match_accuracy": 0.2260094389092816, "lightweight_top8_recall": 0.7194546407970634}
{"epoch": 3, "train_loss": 2.46651443599967, "samples": 1907, "exact_match_accuracy": 0.2548505506030414, "lightweight_top8_recall": 0.7215521761929733}
{"epoch": 4, "train_loss": 2.3034079731229595, "samples": 1907, "exact_match_accuracy": 0.2758259045621395, "lightweight_top8_recall": 0.7320398531725223}
{"epoch": 5, "train_loss": 2.1615218146033603, "samples": 1907, "exact_match_accuracy": 0.27739905610907184, "lightweight_top8_recall": 0.723649711588883}
{"epoch": 6, "train_loss": 2.0105471117807796, "samples": 1907, "exact_match_accuracy": 0.2878867330886209, "lightweight_top8_recall": 0.724698479286838}
{"epoch": 7, "train_loss": 1.872373386084502, "samples": 1907, "exact_matc

## 7. Aggregate five-fold results

In [12]:
import statistics

fold_records = []
for fold in range(5):
    summary_path = DRIVE_RESULTS_ROOT / f"fold_{fold}/training_summary.json"
    if not summary_path.is_file():
        continue
    with summary_path.open("r", encoding="utf-8") as file:
        summary = json.load(file)
    best = max(summary["history"], key=lambda row: row["exact_match_accuracy"])
    fold_records.append({
        "fold": fold,
        "best_epoch": best["epoch"],
        "samples": best["samples"],
        "exact_match_accuracy": best["exact_match_accuracy"],
        "lightweight_top8_recall": best.get("lightweight_top8_recall"),
        "epochs_completed": summary["epochs_completed"],
        "verifier": summary["config"].get("verifier", VERIFIER),
    })

aggregate = {
    "experiment": "full_24class_no_transformer_resident_v1",
    "verifier": VERIFIER,
    "folds": fold_records,
    "completed_fold_count": len(fold_records),
}
if len(fold_records) == 5:
    total_validation_samples = sum(row["samples"] for row in fold_records)
    aggregate["total_validation_samples"] = total_validation_samples
    aggregate["weighted_exact_match_accuracy"] = sum(
        row["samples"] * row["exact_match_accuracy"] for row in fold_records
    ) / total_validation_samples
    selected_final_epochs = max(
        1, int(round(statistics.median(row["best_epoch"] for row in fold_records)))
    )
    aggregate["selected_final_epochs"] = selected_final_epochs
else:
    selected_final_epochs = None
with (DRIVE_RESULTS_ROOT / "five_fold_summary.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(aggregate, file, indent=2)
    file.write("\n")
print(json.dumps(aggregate, indent=2))

{
  "experiment": "full_24class_no_transformer_resident_v1",
  "verifier": "none",
  "folds": [
    {
      "fold": 0,
      "best_epoch": 11,
      "samples": 1907,
      "exact_match_accuracy": 0.32931305715783954,
      "lightweight_top8_recall": 0.6995280545359203,
      "epochs_completed": 15,
      "verifier": "none"
    },
    {
      "fold": 1,
      "best_epoch": 11,
      "samples": 1907,
      "exact_match_accuracy": 0.3361300471945464,
      "lightweight_top8_recall": 0.7409543786051389,
      "epochs_completed": 15,
      "verifier": "none"
    },
    {
      "fold": 2,
      "best_epoch": 19,
      "samples": 1909,
      "exact_match_accuracy": 0.32058669460450495,
      "lightweight_top8_recall": 0.711891042430592,
      "epochs_completed": 20,
      "verifier": "none"
    },
    {
      "fold": 3,
      "best_epoch": 18,
      "samples": 1907,
      "exact_match_accuracy": 0.31253277399056106,
      "lightweight_top8_recall": 0.7147351861562664,
      "epochs_completed"

## 8. Train one final no-Transformer scorer

In [ ]:
from dataclasses import replace

if TRAIN_FINAL_SCORER:
    if selected_final_epochs is None:
        raise RuntimeError("All five folds must finish before final training.")
    final_output_dir = DRIVE_RESULTS_ROOT / "final_scorer"
    final_summary_path = final_output_dir / "training_summary.json"
    if final_summary_path.is_file():
        print(f"final scorer already completed: {final_output_dir}")
    else:
        seed_everything(SEED)
        final_scorer = make_no_transformer_scorer()
        final_config = replace(
            training_config,
            max_epochs=selected_final_epochs,
            patience=selected_final_epochs,
        )
        print(
            f"final no-Transformer training: samples={len(samples)}, "
            f"epochs={selected_final_epochs}, batch={BATCH_SIZE}"
        )
        train_prepared_structured_scorer(
            final_scorer, prepared_samples, [], final_output_dir, final_config
        )
        del final_scorer
        gc.collect()
        torch.cuda.empty_cache()

## 9. Expected outputs

- `results/full_24class_no_transformer_resident_v1/fold_*/best.pt`
- `results/full_24class_no_transformer_resident_v1/five_fold_summary.json`
- `results/full_24class_no_transformer_resident_v1/final_scorer/final.pt`

Compare the weighted five-fold Exact Match directly against EXP-003. Do not combine fold checkpoints. Record measured results in `MODELS.md` only after all five folds complete.